# Notebook 03: Advanced Feature Engineering, Multicollinearity (VIF) & Feature Selection (Module 2)
**Project 02 — Enterprise Predictive Intelligence & Applied Machine Learning System**

This notebook executes our reusable preprocessing and feature engineering pipelines (`src/features/engineering.py`), verifies the cleaned & transformed datasets in `data/processed/`, evaluates Variance Inflation Factors (VIF) before and after pruning, and compares Filter (ANOVA $F$, Mutual Information) and Embedded feature selection rankings.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_PROC = Path("data/processed") if Path("data/processed").exists() else Path("../data/processed")

proc_files = [
    "customer_retention_cleaned_engineered.csv",
    "delivery_delay_cleaned_engineered.csv",
    "daily_product_demand_cleaned_engineered.csv",
    "marketing_funnel_cleaned_engineered.csv",
    "review_sentiment_cleaned_engineered.csv",
    "review_sentiment_unlabeled_predictions.csv",
]

for pf in proc_files:
    df_p = pd.read_csv(DATA_PROC / pf)
    splits = df_p["split"].value_counts().to_dict() if "split" in df_p.columns else {"inference": len(df_p)}
    print(f"{pf:45s} | Shape: {str(df_p.shape):12s} | Nulls: {df_p.isnull().sum().sum()} | Splits: {splits}")


customer_retention_cleaned_engineered.csv     | Shape: (30000, 23)  | Nulls: 0 | Splits: {'train': 21002, 'val': 4499, 'test': 4499}
delivery_delay_cleaned_engineered.csv         | Shape: (40000, 22)  | Nulls: 0 | Splits: {'train': 28002, 'test': 5999, 'val': 5999}
daily_product_demand_cleaned_engineered.csv   | Shape: (105300, 21) | Nulls: 0 | Splits: {'train': 77850, 'val': 13800, 'test': 13650}
marketing_funnel_cleaned_engineered.csv       | Shape: (8000, 17)   | Nulls: 0 | Splits: {'train': 5602, 'val': 1199, 'test': 1199}
review_sentiment_cleaned_engineered.csv       | Shape: (20000, 10)  | Nulls: 0 | Splits: {'train': 14002, 'val': 2999, 'test': 2999}
review_sentiment_unlabeled_predictions.csv    | Shape: (2000, 8)    | Nulls: 0 | Splits: {'inference': 2000}


## 1. Multicollinearity (VIF) Pruning & Multi-Method Feature Selection Audit
We inspect the complete feature selection matrix (`data/processed/feature_selection_and_vif_summary.csv`) combining Pearson $r$, ANOVA $F$-statistic, Mutual Information $I(X;Y)$, initial VIF, post-pruning VIF, and standardized L2 logistic regression coefficients.


In [2]:
df_fs = pd.read_csv(DATA_PROC / "feature_selection_and_vif_summary.csv")
print("=== CUSTOMER RETENTION FEATURE SELECTION & VIF TABLE ===")
print(df_fs[df_fs["task"] == "Customer Retention"].sort_values("mutual_info", ascending=False).to_string(index=False))

print("\n=== DELIVERY DELAY FEATURE SELECTION & VIF TABLE ===")
print(df_fs[df_fs["task"] == "Delivery Delay"].sort_values("mutual_info", ascending=False).to_string(index=False))


=== CUSTOMER RETENTION FEATURE SELECTION & VIF TABLE ===
              task                      feature  pearson_r  anova_f       p_value  mutual_info  vif_initial  vif_selected  selected  std_coef
Customer Retention             orders_last_180d    -0.2585  1503.52 1.003000e-317      0.03574        5.120         4.768      True   -0.4313
Customer Retention                tenure_months    -0.2543  1452.51 2.267000e-307      0.03310        4.813         4.700      True   -0.3056
Customer Retention        days_since_last_order     0.2470  1364.05 2.345000e-289      0.03098        1.001         1.001      True    0.5980
Customer Retention            tickets_per_order     0.1954   833.41 9.463000e-180      0.02284        3.613         3.576      True    0.0389
Customer Retention         engagement_per_order     0.1622   567.40 9.030000e-124      0.01847        6.110           NaN     False    0.0000
Customer Retention order_velocity_per_tenure_yr    -0.0707   105.60  1.033000e-24      0.01